# Mathematical Engineering — Financial Engineering, FY 2025-2026

## Buy Side, Lab 4b — Risk-Based Allocation Strategies

This notebook implements and analyses risk-based allocation strategies on the Euro Stoxx 50 universe with monthly rebalancing and a 2-year rolling estimation window.

**Outline** 

0. Setup & rolling estimators
1. Part I — Equal Risk Contribution (ERC)
2. Part II — Hierarchical Risk Parity (HRP):
   1. Detoning intuition
   2. Cluster stability across rebalances (Rand index)
   3. Sensitivity to clustering choice and to the volatility allocation scheme
3. Part III — Connecting the dots: ERC vs HRP head-to-head and the role of shrinkage


## 0. Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from sklearn.metrics import rand_score

from utilities.backtest import backtest, portfolio_returns
from utilities.covariance_utilities import (
    prepare_rolling_estimation_window,
    covariance_to_correlation,
    risk_contribution,
)
from utilities.hierarchical_clustering import hierarchical_clustering
from utilities.hierarchical_risk_parity import (
    correlation_to_hrp_distance,
    dendrogram_iteration,
    hierarchical_risk_parity,
    recursive_bisection,
)
from utilities.portfolio_optimization import (
    equal_risk_contribution_portfolio,
    inverse_volatility_portfolio,
)
from utilities.principal_component_analysis import (
    detone,
    principal_component_analysis,
)
from utilities.shrinkage import constant_corr_shrinkage, market_factor_shrinkage


SyntaxError: invalid syntax (portfolio_optimization.py, line 50)

In [ ]:
# Read prices, compute daily simple returns
data_path = Path("data")
last_prices = pd.read_csv(
    data_path / "sx5e_underlyings.csv", index_col="Date", parse_dates=True
)
performance = last_prices.pct_change().iloc[1:]

In [ ]:
# Estimation parameters
estimation_window = 2*252  # error (was 252 but need 2y estimation window)
min_coverage = 0.95  # asset must have >=95% non-NaN in the window
trading_days = 252

# Rebalance on the last trading day of each month
rebalance_dates = pd.DatetimeIndex(
    performance.groupby(pd.Grouper(freq="ME"))
    .apply(lambda x: x.index[-1] if len(x) > 0 else None)
    .dropna()
    .values
)


## 1. Pre-computation: rolling covariance estimators

For each rebalance date we estimate three covariance matrices on the trailing 2-year window:

- **sample** — plain `cov()`,
- **constant_corr** — Ledoit–Wolf (2003) constant-correlation shrinkage,
- **mkt_factor** — Ledoit–Wolf (2002) single-factor shrinkage with the
  equally-weighted return of the surviving universe used as the market
  proxy.

The asset universe at every rebalance is restricted to names with
≥95% non-missing observations in the window.


In [ ]:
covariances = {}  # date -> {estimator_name: cov_df}
universes = {}  # date -> [tickers]

for rebalance_date in rebalance_dates:
    cur_returns, diag = prepare_rolling_estimation_window(
        returns=performance,
        rebalance_date=rebalance_date,
        lookback=estimation_window,
        min_coverage=min_coverage,
        return_diagnostics=True,
    )
    if diag["row_count"] < estimation_window or cur_returns.shape[1] == 0:
        continue

    reb = rebalance_date.to_pydatetime().date()
    universes[reb] = list(cur_returns.columns)

    sample_cov = cur_returns.cov()
    constant_corr_cov = constant_corr_shrinkage(cur_returns)["shrunk_cov"]
    market_proxy = cur_returns.mean(axis=1)
    mkt_factor_cov = market_factor_shrinkage(cur_returns, market_proxy)["shrunk_cov"]

    covariances[reb] = {
        "sample": sample_cov,
        "constant_corr": constant_corr_cov,
        "mkt_factor": mkt_factor_cov,
    }

all_dates = sorted(covariances.keys())

## Part I — Equal Risk Contribution (ERC)


### 1.b Sanity check: Risk contribution dispersion & IV ≡ ERC under a diagonal Σ


In [ ]:
iv_erc_delta = {}
erc_normalized_range = {}
for date in all_dates:
    sample_cov = covariances[date]["sample"].values
    
    # 1. Evaluate the dispersion of risk contributions on the real sample covariance
    erc_weights = equal_risk_contribution_portfolio(sample_cov)
    erc_rc = risk_contribution(erc_weights.reshape(1, -1), sample_cov).flatten()
    
    # Calculate normalized range (should be very close to 0 if optimizer converged)
    erc_normalized_range[date] = (
        erc_rc.max() - erc_rc.min()
    ) / np.sum(erc_rc)

    # 2. Verify IV == ERC under a diagonal covariance matrix
    diagonal_cov = np.diag(np.diag(sample_cov))
    iv_weights_diag = inverse_volatility_portfolio(diagonal_cov)
    erc_weights_diag = equal_risk_contribution_portfolio(diagonal_cov)
    
    # Now iv_erc_delta should effectively be 0
    iv_erc_delta[date] = np.linalg.norm(iv_weights_diag - erc_weights_diag, ord=1)

print("--- Part 1.b Sanity Check Results ---")
print(f"Average IV-ERC L1 Delta (Diagonal Sigma): {np.mean(list(iv_erc_delta.values())):.2e}")
print(f"Average ERC Risk Contribution Dispersion: {np.mean(list(erc_normalized_range.values())):.2e}")

### 1.c ERC backtest across the three covariance estimators


In [ ]:
erc_portfolios = {"sample": {}, "constant_corr": {}, "mkt_factor": {}}

for reb in all_dates:
    for est_name, cov in covariances[reb].items():
        erc_portfolios[est_name][reb] = pd.Series(
            equal_risk_contribution_portfolio(cov.values),
            index=cov.index,
            name=reb,
        )

for est_name in erc_portfolios:
    # Get the universe (assets) for each rebalance date to align properly
    portfolio_list = []
    for reb, series in sorted(erc_portfolios[est_name].items()):
        # Align each series to the full performance.columns, filling missing with 0
        aligned = series.reindex(performance.columns, fill_value=0.0)
        portfolio_list.append(aligned)
    # Stack into DataFrame with rebalance dates as rows
    erc_portfolios[est_name] = pd.DataFrame(portfolio_list, index=sorted(erc_portfolios[est_name].keys()))

erc_backtests = {
    est_name: backtest(portfolios, performance)
    for est_name, portfolios in erc_portfolios.items()
}

erc_backtests = pd.DataFrame(erc_backtests)

plt.figure(figsize=(12, 6))
erc_backtests.plot(ax=plt.gca())
plt.title("ERC Cumulative Performance across Estimators")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.show()

summary_stats = pd.DataFrame(index=erc_backtests.columns)

# 1. Cumulative Return
summary_stats['Total Return'] = erc_backtests.iloc[-1] - 1

# 2. 63d EWM Volatility (Annualized)
# Note: we use returns here, not the cumulative backtest
returns_df = erc_backtests.pct_change()
ann_factor = np.sqrt(252)
summary_stats['Avg 63d EWM Vol'] = returns_df.ewm(span=63).std().mean() * ann_factor

# 3. Maximum Drawdown
rolling_max = erc_backtests.cummax()
drawdowns = (erc_backtests - rolling_max) / rolling_max
summary_stats['Max Drawdown'] = drawdowns.min()

# 4. Average Turnover
# Computed from the weight dataframes created in Cell 7
for est_name in erc_portfolios:
    weights = erc_portfolios[est_name]
    turnover = weights.diff().abs().sum(axis=1).mean()
    summary_stats.loc[est_name, 'Avg Monthly Turnover'] = turnover

print("--- Part 1.c ERC Performance Summary ---")
print(summary_stats)

In [ ]:
# DEBUG: Check shrinkage targets and how they differ from sample
print("=== DEBUGGING: Shrinkage Targets ===")
first_date = all_dates[0]

# Get the returns for first rebalance date
cur_returns, _ = prepare_rolling_estimation_window(
    returns=performance,
    rebalance_date=pd.Timestamp(first_date),
    lookback=estimation_window,
    min_coverage=min_coverage,
    return_diagnostics=True,
)

sample_cov = cur_returns.cov().values
N = sample_cov.shape[0]

# Extract correlations
sample_std = np.sqrt(np.diag(sample_cov))
sample_outer = np.outer(sample_std, sample_std)
sample_corr = sample_cov / sample_outer

cc_result = constant_corr_shrinkage(cur_returns)
print(f"\n--- Constant Corr Shrinkage ---")
print(f"Intensity: {cc_result['intensity']:.6f}")
cc_target = cc_result['target'].values
cc_corr = cc_target / np.outer(np.sqrt(np.diag(cc_target)), np.sqrt(np.diag(cc_target)))
cc_shrunk = cc_result['shrunk_cov'].values

# Compute average correlation
avg_sample_corr = (np.sum(sample_corr) - N) / (N * (N - 1))
avg_cc_corr = (np.sum(cc_corr) - N) / (N * (N - 1))
print(f"Sample avg correlation: {avg_sample_corr:.6f}")
print(f"CC Target avg correlation: {avg_cc_corr:.6f}")
print(f"Sample vs CC Target Max Diff: {np.max(np.abs(sample_cov - cc_target)):.6f}")
print(f"Sample vs CC Shrunk Max Diff: {np.max(np.abs(sample_cov - cc_shrunk)):.6f}")

market_proxy = cur_returns.mean(axis=1)
mf_result = market_factor_shrinkage(cur_returns, market_proxy)
print(f"\n--- Market Factor Shrinkage ---")
print(f"Intensity: {mf_result['intensity']:.6f}")
mf_target = mf_result['target'].values
mf_shrunk = mf_result['shrunk_cov'].values

# Compute betas and residuals
mkt_var = market_proxy.var()
betas = np.array([cur_returns[col].cov(market_proxy) / mkt_var for col in cur_returns.columns])
residual_vars = cur_returns.var().values - mkt_var * betas**2
print(f"Market Variance: {mkt_var:.6f}")
print(f"Avg Beta: {betas.mean():.6f}")
print(f"Avg Residual Var: {residual_vars.mean():.6f}")
print(f"Sample vs MF Target Max Diff: {np.max(np.abs(sample_cov - mf_target)):.6f}")
print(f"Sample vs MF Shrunk Max Diff: {np.max(np.abs(sample_cov - mf_shrunk)):.6f}")

print(f"\n--- Between Targets ---")
print(f"CC Target vs MF Target Max Diff: {np.max(np.abs(cc_target - mf_target)):.6f}")

print(f"\n--- Between Shrunk Covs ---")
print(f"CC Shrunk vs MF Shrunk Max Diff: {np.max(np.abs(cc_shrunk - mf_shrunk)):.6f}")

## Part II — Hierarchical Risk Parity (HRP)


### 2.a Detoning — intuition


In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!

### 2.c Cluster stability across rebalances (Rand index)

We measure how stable the clustering is between consecutive rebalances via the Rand index. A value close to 1 means the partition into $k$ clusters changes little month-to-month.


In [ ]:
LINKAGE_METHOD = "ward"
DIST_METRIC = "euclidean"
N_CLUSTERS_GRID = [5, 8, 10, 15]
N_CLUSTERS_FOCUS = 10
COV_KEYS = ["sample", "constant_corr", "mkt_factor"]
DETONE_KEYS = [False, True]


linkages = {(c, d): {} for c in COV_KEYS for d in DETONE_KEYS}
clusters = {
    (c, d, k): {} for c in COV_KEYS for d in DETONE_KEYS for k in N_CLUSTERS_GRID
}

for reb in all_dates:
    for cov_key in COV_KEYS:
        cov = covariances[reb][cov_key]
        for detone_flag in DETONE_KEYS:
            link = None  # !!! COMPLETE AS APPROPRIATE !!!
            linkages[(cov_key, detone_flag)][reb] = link
            for n_clusters in N_CLUSTERS_GRID:
                clusters[(cov_key, detone_flag, n_clusters)][reb] = pd.Series(
                    sch.cut_tree(link.values, n_clusters=n_clusters).flatten(),
                    index=cov.index,
                )

# !!! COMPLETE AS APPROPRIATE !!!

### 2.d Sensitivity of HRP weights — clustering vs. allocation traversal


In [ ]:
LINK_GRID = ["single", "ward", "complete", "average"]
ALLOC_GRID = {
    "recursive_bisection": recursive_bisection,
    "dendrogram_iteration": dendrogram_iteration,
}

hrp_grid = {}
for cov_key in COV_KEYS:
    for link in LINK_GRID:
        for alloc_name, alloc in ALLOC_GRID.items():
            for d in DETONE_KEYS:
                hrp_grid[(cov_key, link, alloc_name, d)] = (
                    None  # !!! COMPLETE AS APPROPRIATE !!!
                )

# !!! COMPLETE AS APPROPRIATE !!!

## Part III — Connecting the Dots


### 3.a ERC vs HRP — Robustness Against Estimator Choice

In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!

### 3.b ERC vs HRP — performance, vol, drawdown, turnover


In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!